# 17 · Failure Case 분석 ⭐

**학술 깊이 보강** — 평가 항목 3번 (성능 분석) + 4번 (시각화) 동시 강화.

**목적**: 
- val set에서 **sample-wise mIoU** 계산
- **가장 못 맞춘 5개 샘플** 추출
- **실패 원인 분석** (어디서 틀렸나? 왜?)
- **시각화**: Input / GT / Pred / Error map

**산출물**:
- `outputs/failure_cases/worst_5_samples.png` — 발표 핵심 슬라이드
- `failure_analysis.csv` — sample별 mIoU + 통계

## 1. 환경 + 모델 로드 + 데이터셋

In [ ]:
import os, sys, subprocess
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/cv-final'
    if not os.path.exists(REPO):
        from google.colab import userdata
        token = userdata.get('GH_TOKEN').strip()
        result = subprocess.run(
            ['git', 'clone', f'https://{token}@github.com/keonU206/cv-final.git', REPO],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(result.stderr)
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    !pip install -q segmentation_models_pytorch opencv-python pyyaml datasets pandas

print(f'CWD: {os.getcwd()}')

In [ ]:
import torch
import numpy as np, cv2, matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from project.segmentation.unet import build_unet
from project.segmentation.data import CelebAMaskHQDataset
from project.segmentation.transforms import get_val_transform
from project.segmentation.evaluation import compute_confusion_matrix, per_class_iou
from torch.utils.data import DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('outputs/failure_cases')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['bg', 'nose', 'eye', 'mouth', 'skin', 'unused']
NUM_CLASSES = 6
INPUT_SIZE = 256
VAL_SUBSET = 50  # 분석 대상

# Attention U-Net 로드 (가장 성능 좋음)
MODEL_CKPT = '/content/drive/MyDrive/cv-final-ckpts/unet_attention_best.pt'
model = build_unet(
    num_classes=NUM_CLASSES,
    encoder_name='resnet34', encoder_weights=None,
    in_channels=3, attention_type='scse',
).to(device)
model.load_state_dict(torch.load(MODEL_CKPT, map_location=device))
model.eval()
print(f'✅ Attention U-Net 로드')

In [ ]:
val_tf = get_val_transform(size=INPUT_SIZE, with_heatmap=False)
val_ds = CelebAMaskHQDataset(
    split='val', transform=val_tf,
    subset_size=VAL_SUBSET,
    cache_dir='/content/data/celebamask',
    input_size=INPUT_SIZE,
    use_landmark=False,
)
val_dl = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)
print(f'✅ Val: {len(val_ds)}')

## 2. Sample-wise mIoU 측정

In [ ]:
# sample별 mIoU 계산
sample_results = []  # [(idx, mIoU, img, gt, pred), ...]

with torch.no_grad():
    for idx, batch in enumerate(val_dl):
        img, gt = batch  # img: (1,3,H,W), gt: (1,H,W)
        img, gt = img.to(device), gt.to(device)
        pred = model(img)  # (1, 6, H, W)
        pred_cls = pred.argmax(dim=1)  # (1, H, W)
        
        # 이 샘플의 confusion matrix + mIoU
        conf = compute_confusion_matrix(pred, gt, NUM_CLASSES)
        ious = per_class_iou(conf)
        # background 제외 mIoU (실제 분석 가치 있는 클래스)
        valid_ious = ious[1:5]  # nose, eye, mouth, skin
        sample_miou = float(valid_ious.mean())
        
        img_np = (img[0].cpu().numpy().transpose(1, 2, 0) * 255).astype(np.uint8)
        gt_np = gt[0].cpu().numpy()
        pred_np = pred_cls[0].cpu().numpy()
        
        sample_results.append({
            'idx': idx,
            'mIoU': sample_miou,
            'per_class_iou': ious.tolist(),
            'image': img_np,
            'gt': gt_np,
            'pred': pred_np,
        })

print(f'✅ {len(sample_results)}개 샘플 평가 완료')

# mIoU 기준 정렬
sample_results_sorted = sorted(sample_results, key=lambda x: x['mIoU'])

# 통계
all_mious = [s['mIoU'] for s in sample_results]
print(f'\nmIoU 통계 (background 제외):')
print(f'  평균: {np.mean(all_mious):.4f}')
print(f'  중앙값: {np.median(all_mious):.4f}')
print(f'  표준편차: {np.std(all_mious):.4f}')
print(f'  최저: {min(all_mious):.4f}')
print(f'  최고: {max(all_mious):.4f}')

## 3. mIoU 분포 히스토그램

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(all_mious, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(np.mean(all_mious), color='red', linestyle='--', label=f'Mean: {np.mean(all_mious):.3f}')
ax.axvline(np.median(all_mious), color='green', linestyle='--', label=f'Median: {np.median(all_mious):.3f}')
ax.set_xlabel('Sample mIoU (nose/eye/mouth/skin)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Sample-wise mIoU Distribution', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
save_hist = OUTPUT_DIR / 'miou_distribution.png'
plt.savefig(save_hist, dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✅ 저장: {save_hist}')

## 4. Worst-5 Failure Cases 시각화 ⭐ (발표 핵심)

In [ ]:
# 시각화 색상 매핑 (클래스별)
CLASS_COLORS = np.array([
    [0, 0, 0],         # 0: background (검정)
    [255, 100, 100],   # 1: nose (빨강)
    [100, 255, 100],   # 2: eye (초록)
    [100, 100, 255],   # 3: mouth (파랑)
    [255, 200, 100],   # 4: skin (주황)
    [200, 100, 200],   # 5: unused (보라)
], dtype=np.uint8)

def mask_to_rgb(mask):
    """(H, W) class indices → (H, W, 3) RGB."""
    return CLASS_COLORS[mask]

def error_map(gt, pred):
    """(H, W) GT vs Pred → 빨간색 = 틀린 픽셀."""
    err = (gt != pred).astype(np.uint8)
    rgb = np.zeros((*gt.shape, 3), dtype=np.uint8)
    rgb[err == 1] = [255, 0, 0]  # 빨강 = 오답
    return rgb

# Worst 5 시각화 (5 row × 4 col)
worst_5 = sample_results_sorted[:5]
fig, axes = plt.subplots(5, 4, figsize=(16, 20))

for i, sample in enumerate(worst_5):
    axes[i, 0].imshow(sample['image'])
    axes[i, 0].set_title(f'Input\nidx={sample["idx"]}', fontsize=11)
    
    axes[i, 1].imshow(mask_to_rgb(sample['gt']))
    axes[i, 1].set_title('Ground Truth', fontsize=11)
    
    axes[i, 2].imshow(mask_to_rgb(sample['pred']))
    axes[i, 2].set_title(f'Prediction\nmIoU={sample["mIoU"]:.3f}', fontsize=11)
    
    # 입력 + 오답 영역 overlay
    err = error_map(sample['gt'], sample['pred'])
    overlay = (0.6 * sample['image'] + 0.4 * err).astype(np.uint8)
    axes[i, 3].imshow(overlay)
    err_ratio = (sample['gt'] != sample['pred']).mean()
    axes[i, 3].set_title(f'Error Map\n오답 {err_ratio:.1%}', fontsize=11)
    
    for ax in axes[i]:
        ax.axis('off')

fig.suptitle('Worst 5 Failure Cases — 모델이 가장 못 맞춘 샘플', fontsize=15, y=0.995)
plt.tight_layout()
save_worst = OUTPUT_DIR / 'worst_5_samples.png'
plt.savefig(save_worst, dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✅ 저장: {save_worst}')

## 5. Best 3 vs Worst 3 비교 (성공/실패 패턴)

In [ ]:
best_3 = sample_results_sorted[-3:][::-1]  # 가장 잘한 3개
worst_3 = sample_results_sorted[:3]

fig, axes = plt.subplots(2, 6, figsize=(24, 8))

# Best 3 (윗줄)
for i, sample in enumerate(best_3):
    axes[0, i*2].imshow(sample['image'])
    axes[0, i*2].set_title(f'Best {i+1} · idx={sample["idx"]}', fontsize=11)
    axes[0, i*2+1].imshow(mask_to_rgb(sample['pred']))
    axes[0, i*2+1].set_title(f'Pred · mIoU={sample["mIoU"]:.3f}', fontsize=11)

# Worst 3 (아랫줄)
for i, sample in enumerate(worst_3):
    axes[1, i*2].imshow(sample['image'])
    axes[1, i*2].set_title(f'Worst {i+1} · idx={sample["idx"]}', fontsize=11)
    axes[1, i*2+1].imshow(mask_to_rgb(sample['pred']))
    axes[1, i*2+1].set_title(f'Pred · mIoU={sample["mIoU"]:.3f}', fontsize=11)

for ax in axes.flatten():
    ax.axis('off')

fig.suptitle('Best 3 vs Worst 3 — 성공/실패 패턴 비교', fontsize=15, y=1.02)
plt.tight_layout()
save_compare = OUTPUT_DIR / 'best_vs_worst.png'
plt.savefig(save_compare, dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✅ 저장: {save_compare}')

## 6. 실패 원인 분석 (Per-class IoU 패턴)

In [ ]:
# Worst 5의 per-class IoU 평균 vs Best 5
best_5 = sample_results_sorted[-5:]
worst_5 = sample_results_sorted[:5]

best_class_ious = np.array([s['per_class_iou'] for s in best_5]).mean(axis=0)
worst_class_ious = np.array([s['per_class_iou'] for s in worst_5]).mean(axis=0)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(NUM_CLASSES)
w = 0.35
ax.bar(x - w/2, best_class_ious, w, label='Best 5 (avg)', color='green', alpha=0.7)
ax.bar(x + w/2, worst_class_ious, w, label='Worst 5 (avg)', color='red', alpha=0.7)

for i, (b, w_v) in enumerate(zip(best_class_ious, worst_class_ious)):
    ax.text(i - 0.18, b + 0.01, f'{b:.2f}', ha='center', fontsize=9)
    ax.text(i + 0.18, w_v + 0.01, f'{w_v:.2f}', ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel('IoU', fontsize=12)
ax.set_title('Per-class IoU · Best vs Worst Samples', fontsize=14)
ax.legend(loc='upper right')
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
save_class = OUTPUT_DIR / 'best_vs_worst_per_class.png'
plt.savefig(save_class, dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✅ 저장: {save_class}')

# 어떤 클래스가 worst case에서 가장 안 좋아지나?
delta = best_class_ious - worst_class_ious
print('\n=== 실패 패턴 ===')
for i, cname in enumerate(CLASS_NAMES):
    print(f'  {cname:<8}: Best {best_class_ious[i]:.3f} → Worst {worst_class_ious[i]:.3f}  (Δ={delta[i]:+.3f})')

## 7. CSV 저장 + Drive 백업

In [ ]:
# CSV 저장 (모든 샘플 mIoU)
df_data = []
for s in sample_results:
    row = {'idx': s['idx'], 'mIoU': s['mIoU']}
    for i, cname in enumerate(CLASS_NAMES):
        row[f'IoU_{cname}'] = s['per_class_iou'][i]
    df_data.append(row)

df = pd.DataFrame(df_data)
df = df.sort_values('mIoU')
save_csv = OUTPUT_DIR / 'failure_analysis.csv'
df.to_csv(save_csv, index=False)
print(f'\n✅ CSV 저장: {save_csv}')
print('\n=== Worst 5 ===')
print(df.head(5).to_string(index=False))
print('\n=== Best 5 ===')
print(df.tail(5).to_string(index=False))

# Drive 백업
if IS_COLAB:
    DRIVE_OUT = Path('/content/drive/MyDrive/cv-final-ckpts/samples')
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    !cp -v {OUTPUT_DIR}/*.png {OUTPUT_DIR}/*.csv {DRIVE_OUT}/ 2>&1 | tail -10
    print(f'\n✅ Drive 백업: {DRIVE_OUT}/')

---

## 📚 학술 해석 (발표 슬라이드 멘트)

**Failure Case 분석 의의**:
1. **모델이 어디서 틀리는가** — 단순 mIoU 평균이 가리는 한계 노출
2. **클래스별 난이도** 정량적 검증
3. **데이터 다양성 한계** 발견 (특이 포즈, 조명, 색상 등)

**예상 실패 패턴**:
- 측면 얼굴 (정면 학습 데이터)
- 안경, 머리카락이 눈/입을 가림
- 강한 조명 / 그림자
- 메이크업, 헤어밴드 등 outlier 액세서리

**해결 방향** (향후 강화 슬라이드 #10):
1. **Heavy augmentation** — CutMix, Mosaic, RandAug
2. **Hard example mining** — worst N% 다시 학습
3. **Multi-task learning** — landmark + segmentation 동시 학습

**평가 항목 매핑**:
- **항목 3번 (성능 분석 15%)**: 단순 평균이 아닌 sample-wise 분석
- **항목 4번 (시각화 15%)**: Error Map → "어디서 틀렸는지" 시각화
- **항목 1번 (문제 정의 30%)**: 데이터 한계 인지 = 문제 이해 깊이

**참고문헌**:
- Lin, T. Y. et al. (2017). *Focal Loss for Dense Object Detection*. ICCV. (Hard example)
- He, T. et al. (2019). *Bag of Tricks for Image Classification*. CVPR. (Augmentation)
- Yun, S. et al. (2019). *CutMix: Regularization Strategy*. ICCV.

**발표 멘트 예시**:
> "단순 평균 mIoU 외에 **Failure Case 분석**으로 모델 한계를 정량 검증했습니다.
> Worst 5 샘플은 mIoU **0.XX 이하**로 평균 대비 **YY% 낮음**.
> 공통 패턴: **측면 얼굴, 안경 가림, 강한 조명** 등 학습 데이터 다양성 부족.
> 향후 **CutMix** (Yun 2019 ICCV)와 **hard example mining** (Lin 2017)으로 강화 가능합니다."